# conv-output-shape composite — cx16: 2D conv with stride+padding: predict shape, then build window view

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-output-shape`, `conv-windowing-2d`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-output-shape"
DD_ATOM_IDS = ["conv-output-shape", "conv-windowing-2d"]
DD_SUBTOPICS = ["CNN: Conv output shape", "CNN: 2-D conv windowing"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The 2-D analogue of cx15. The output-shape formula now produces TWO output dims:

```
OH = (H + 2*PH - KH) // SH + 1
OW = (W + 2*PW - KW) // SW + 1
```

Once you have `OH, OW`, the 2-D windowing trick builds the 6-axis view `(B, IC, OH, OW, KH, KW)`. With stride and padding, the OH/OW strides become `s_h * SH` and `s_w * SW` respectively; the KH/KW strides stay at `s_h, s_w`.

**Padding for 2-D.** `F.pad(x, (PW, PW, PH, PH))` — last-axis padding comes first in the tuple (it's PyTorch's convention, NOT a typo). After padding, read `padded.stride()` for the fresh `(s_b, s_ic, s_h, s_w)`.

**Why decouple shape from view.** ARENA's modular Conv2d class computes the output shape BEFORE building the view — because shape errors must surface as predictable assertions, not as a downstream as_strided crash with an uninformative message.

### Composite Exercise — 2D conv with stride+padding: predict shape, then build window view

**Atoms exercised together**: `conv-output-shape`, `conv-windowing-2d`

Implement `cx16_strided_conv2d(x, weight, stride=(1,1), padding=(0,0))`.

- `x`: float tensor `(B, IC, H, W)`.
- `weight`: float tensor `(OC, IC, KH, KW)`.
- `stride`, `padding`: 2-tuples of ints.
- Return: tensor `(B, OC, OH, OW)` matching `F.conv2d(x, weight, stride=stride, padding=padding)`.

Also implement `cx16_predict_outshape(input_shape, OC, kernel_size, stride, padding)`. The test verifies the predicted shape matches the F.conv2d output shape on a battery of cases.

**Tip.** For F.pad on 4-D tensors, the order is `(left, right, top, bottom)` — last axis padding first. Easy mistake: passing `(PH, PH, PW, PW)` swaps height and width.

In [ ]:
def cx16_predict_outshape(input_shape, OC, kernel_size, stride, padding):
    # Atom A (conv-output-shape, 2-D form).
    B, IC, H, W = input_shape
    KH, KW = kernel_size
    SH, SW = stride
    PH, PW = padding
    OH = (H + 2 * PH - KH) // SH + 1
    OW = (W + 2 * PW - KW) // SW + 1
    return (B, OC, OH, OW)

def cx16_strided_conv2d(x, weight, stride=(1,1), padding=(0,0)):
    from torch.nn import functional as F
    B, IC, H, W = x.shape
    OC, IC2, KH, KW = weight.shape
    assert IC == IC2
    SH, SW = stride
    PH, PW = padding
    _, _, OH, OW = cx16_predict_outshape(x.shape, OC, (KH, KW), stride, padding)
    # Pad: F.pad on 4-D wants (left, right, top, bottom) — last-axis first.
    xp = F.pad(x, (PW, PW, PH, PH)) if (PH > 0 or PW > 0) else x
    s_b, s_ic, s_h, s_w = xp.stride()
    # Atom B (conv-windowing-2d): step SH/SW on the OH/OW axes.
    x_win = xp.as_strided(
        size=(B, IC, OH, OW, KH, KW),
        stride=(s_b, s_ic, s_h * SH, s_w * SW, s_h, s_w),
    )
    return einops.einsum(
        x_win, weight,
        'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow',
    )


<details><summary>Show solution — cx16</summary>

```python
def cx16_predict_outshape(input_shape, OC, kernel_size, stride, padding):
    # Atom A (conv-output-shape, 2-D form).
    B, IC, H, W = input_shape
    KH, KW = kernel_size
    SH, SW = stride
    PH, PW = padding
    OH = (H + 2 * PH - KH) // SH + 1
    OW = (W + 2 * PW - KW) // SW + 1
    return (B, OC, OH, OW)

def cx16_strided_conv2d(x, weight, stride=(1,1), padding=(0,0)):
    from torch.nn import functional as F
    B, IC, H, W = x.shape
    OC, IC2, KH, KW = weight.shape
    assert IC == IC2
    SH, SW = stride
    PH, PW = padding
    _, _, OH, OW = cx16_predict_outshape(x.shape, OC, (KH, KW), stride, padding)
    # Pad: F.pad on 4-D wants (left, right, top, bottom) — last-axis first.
    xp = F.pad(x, (PW, PW, PH, PH)) if (PH > 0 or PW > 0) else x
    s_b, s_ic, s_h, s_w = xp.stride()
    # Atom B (conv-windowing-2d): step SH/SW on the OH/OW axes.
    x_win = xp.as_strided(
        size=(B, IC, OH, OW, KH, KW),
        stride=(s_b, s_ic, s_h * SH, s_w * SW, s_h, s_w),
    )
    return einops.einsum(
        x_win, weight,
        'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow',
    )
```

The OH/OW axes get multiplied strides (`s_h * SH`, `s_w * SW`) because incrementing the output position by 1 corresponds to moving `S` input cells. The KH/KW axes do NOT get multiplied — within a window, you always read every adjacent input position.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx16'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx16',
        'subtopics': ["CNN: Conv output shape", "CNN: 2-D conv windowing"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()